In [1]:
import torch    
import torch_geometric.transforms as T
import mdtraj as md
import os
import torch 
import numpy as np
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data, Dataset
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import BatchNorm1d
from torch_geometric.nn import GATConv, global_mean_pool, GCNConv, knn_graph, Linear
from torch.optim import SGD, Adam, Optimizer
import math
from torch.nn.init import kaiming_uniform_
from torch_geometric.transforms import ToDevice
import scipy
import sys
from e3nn.io import CartesianTensor
from torch_geometric.loader import DataLoader

device='cuda'
import torch
torch.cuda.is_available()

# print(torch.version.cuda)


True

In [4]:
# top = "an1.gro" 
top ='../../PHD2_100_O2IF/an1.gro'
t = "../../PHD2_100_O2IF/Sim10/cat_small.dcd"
traj = md.load(t, top=top)

/home/coyote/miniconda3/envs/holoprot/lib/python3.10/site-packages/mdtraj/formats/gro.py:364: UserWarning: WARNING: two consecutive residues with same number (GLN, ACE)
  warnings.warn(
/home/coyote/miniconda3/envs/holoprot/lib/python3.10/site-packages/mdtraj/formats/gro.py:364: UserWarning: WARNING: two consecutive residues with same number (ASP, ACE)
  warnings.warn(


In [5]:
from rdkit import Chem
mol = Chem.MolFromPDBFile("../../PHD2_100_O2IF/an1.pdb", removeHs=False)
Chem.SanitizeMol(mol)
for bond in mol.GetBonds():
    print(bond.GetBeginAtom().GetSymbol(), bond.GetBeginAtom().GetIdx(),
          bond.GetEndAtom().GetSymbol(), bond.GetEndAtom().GetIdx(),
          bond.GetBondType())



H 1 C 0 SINGLE
H 2 C 0 SINGLE
H 3 C 0 SINGLE
C 4 C 0 SINGLE
O 5 C 4 DOUBLE
N 6 C 4 SINGLE
H 7 N 6 SINGLE
C 8 N 6 SINGLE
H 9 C 8 SINGLE
C 10 C 8 SINGLE
H 11 C 10 SINGLE
H 12 C 10 SINGLE
C 13 C 10 SINGLE
H 14 C 13 SINGLE
H 15 C 13 SINGLE
C 16 C 13 SINGLE
O 17 C 16 DOUBLE
N 18 C 16 SINGLE
H 19 N 18 SINGLE
H 20 N 18 SINGLE
C 21 C 8 SINGLE
O 22 C 21 DOUBLE
N 23 C 21 SINGLE
H 24 N 23 SINGLE
C 25 N 23 SINGLE
H 26 C 25 SINGLE
C 27 C 25 SINGLE
H 28 C 27 SINGLE
O 29 C 27 SINGLE
H 30 O 29 SINGLE
C 31 C 27 SINGLE
H 32 C 31 SINGLE
H 33 C 31 SINGLE
H 34 C 31 SINGLE
C 35 C 25 SINGLE
O 36 C 35 DOUBLE
N 37 C 35 SINGLE
H 38 N 37 SINGLE
C 39 N 37 SINGLE
H 40 C 39 SINGLE
C 41 C 39 SINGLE
H 42 C 41 SINGLE
H 43 C 41 SINGLE
C 44 C 41 SINGLE
H 45 C 44 SINGLE
H 46 C 44 SINGLE
C 47 C 44 SINGLE
H 48 C 47 SINGLE
H 49 C 47 SINGLE
C 50 C 47 SINGLE
H 51 C 50 SINGLE
H 52 C 50 SINGLE
N 53 C 50 SINGLE
H 54 N 53 SINGLE
H 55 N 53 SINGLE
H 56 N 53 SINGLE
C 57 C 39 SINGLE
O 58 C 57 DOUBLE
N 59 C 57 SINGLE
C 60 N 59 SINGLE


In [ ]:
# protein_cofactors = torch.tensor(np.setdiff1d(np.arange(0,xyz.shape[1]), gas_atoms[gas_idx])).to(device)

NameError: name 'xyz' is not defined

In [7]:
xyz = torch.tensor(traj.xyz) * 10
ele2num = {"C": 0, "H": 1, "O": 2, "N": 3, "S": 4, "VS": {"FE": 5, "MG":6}} # all 0's will be padding for gases not within 3.5 angstroms of any protein atom
gas='O2IF'
metal='FE'
gas2 = traj.topology.select('resname %s' % gas)
residue_ref = np.array([traj.topology.atom(ind).residue.resSeq for ind in gas2])
FE = traj.topology.select('resname Fe2p')
residue_sel_un = np.unique(residue_ref) # gas
residue_sel_un

print('LEN FE', FE)
residue_sel_un = np.unique(residue_ref) # gas
residue_sel_un
nogas = np.setdiff1d(range(0,traj.xyz.shape[1]),gas2)
rnames = np.array([traj.topology.atom(ind).residue.name for ind in nogas])
rindex = np.array([traj.topology.atom(ind).residue.resSeq for ind in nogas])
anames = np.array([traj.topology.atom(ind).element.symbol for ind in nogas])
anames2=np.array([traj.topology.atom(i).name for i in nogas])
anums = [ele2num[a] if a != 'VS' else ele2num[a][metal] for a in anames]

rnames = np.array([traj.topology.atom(ind).residue.name for ind in nogas])
rindex = np.array([traj.topology.atom(ind).residue.resSeq for ind in nogas])
anames = np.array([traj.topology.atom(ind).element.symbol for ind in nogas])
rnames2 = np.array([traj.topology.atom(ind).residue for ind in nogas])

cat = np.where(anames=='VS')[0]
cat

device = torch.device("cpu")
# device = torch.device("cpu")
protein_coords_traj = xyz[:,nogas,:]
rs = int(len(residue_ref)/len(residue_sel_un))
gas_atoms=gas2.reshape(len(residue_sel_un),2)

dioxygen_coords_ave = xyz[:,gas2,:].reshape(-1,len(residue_sel_un),rs,3).mean(axis=2)
d=torch.cdist(dioxygen_coords_ave, xyz[:,cat,:])
diox = np.where(d < 6.0)[1]
frames = np.where(d < 6.0)[0]

LEN FE [3790]


In [8]:
types_array_atom = torch.zeros((len(nogas)+len(gas2), (len(ele2num))))
for i, t in enumerate(anums):
    types_array_atom[i,t] = 1.0
# types_array_atom.shape
types_array_atom[-len(gas2):,ele2num['O']] = 1
types_array_atom = types_array_atom.to(device)

4090

In [9]:
# types_array_atom.shape
from torch_geometric.data import Data
def extract_point_cloud(atom_matrix, positions, center):
    """
    Formats the already-cropped atom features and positions into a Data object.
    """
    if positions.shape[0] == 0:
        return None  # Skip empty cubes

    # Center coordinates relative to cube center
    centered_positions = positions - center

    return Data(x=atom_matrix, pos=centered_positions)

path_dict = {
    'P1':'P1', 
    'P_main (PmR)':'PmR', 
    'P_main (mid)':'mid', 
    'P_reverse':'P_reverse', 
    'P_main (PmL)':'PmL',
       'P3':'P3'
}

def write_pdb2(inds, what, xs, ys, zs, chain='C', file="ml_out.pdb"):
    
    print(file)
    
    fpdb = open(file, 'wt')
    norm = torch.max(what[inds])
    i_atom = 1
    i_resid = 1
    for i, dind in enumerate(inds):

        fpdb.write('{:6s}{:5d} {:^4s}{:1s}{:3s} {:1s}{:4d}{:1s}   {:8.3f}{:8.3f}{:8.3f}{:6.2f}{:6.2f}          {:>2s}{:2s}\n'.format(
            'ATOM',i_atom,
            'GG','','GG',
            chain,i_resid,'',
            xs[dind],ys[dind],zs[dind],
            -1.0*torch.log10(what[dind]/norm),(what[dind]),
            'K',''))
        i_atom += 1
        if i_atom > 999:
            i_atom = 1
            i_resid += 1
            #eigv[dind,0],mm.pi[dind],
    fpdb.write('TER\n')
    fpdb.close()

def get_points(start, translated, frame_pos, space=0.65, bottom_threshold=2.5):
    x_edges = np.arange(-20, 21, space)
    y_edges = np.arange(-20, 21, space)
    z_edges = np.arange(-20, 21, space)

    X, Y, Z = np.meshgrid(x_edges, y_edges, z_edges, indexing='ij')

    # Stack the arrays to create a 3D array of shape (N, 3)
    points_3d = torch.tensor(np.stack([X, Y, Z], axis=-1).reshape(-1, 3), dtype=torch.float32)

    translated_ang = translated.clone()
    exclude_points = translated_ang

    threshold = bottom_threshold
    threshold2 = 3.5

    # Compute distances from every point in `points_3d` to every `exclude_point`
    distances=torch.cdist(points_3d, exclude_points)

    # Find points in `points_3d` with all distances >= threshold
    mask = torch.all(distances >= threshold, axis=1)
    mask2 = torch.any(distances < threshold2, axis=1)

    # Filter `points_3d` to keep only points outside the threshold
    filtered_points = points_3d[mask & mask2]

    filtered_points2 = filtered_points# + protein_coords_list.cpu()[cat].numpy() 
    filtered_points2.mean(axis=0)
    filtered_points2.shape
    dist2 = torch.norm(filtered_points2-frame_pos, dim=1)
    if start==0:
        filtered_points3 = filtered_points2[dist2 <= 20]
    else:
        filtered_points3 = filtered_points2[dist2 <= 5]
    xyz2=filtered_points3

    return xyz2

def get_fe_bias(xyz2, frame_pos, alpha=10.0, beta=1.0, denom=4, range_=0.75):
    """
    alpha → dominance of origin distance
    beta  → influence of closeness to the line
    """

    # ------------------------------
    # Distance to origin (dominant term)
    # ------------------------------
    dist_origin = torch.norm(xyz2, dim=1)  # [N]
    # Invert + square for extreme emphasis (close gets very large)
    origin_score = 1.0 / (dist_origin**denom + 1e-8)

    # ------------------------------
    # Distance to line
    # ------------------------------
    line_dir = frame_pos / torch.norm(frame_pos)
    t = (xyz2 * line_dir).sum(dim=1, keepdim=True)
    d_perp = torch.norm(xyz2 - t * line_dir, dim=1)

    # Invert perpendicular distance
    line_score = 1.0 / (d_perp + 1e-8)

    # ------------------------------
    # Combine WITHOUT global normalization
    # (so origin dominance is preserved)
    # ------------------------------
    combined = alpha * origin_score + beta * line_score

    # ------------------------------
    # Scale result to 0–0.5 while preserving ranking
    # ------------------------------
    combined_min = combined.min()
    combined_max = combined.max()

    score = range_ * (combined - combined_min) / (combined_max - combined_min + 1e-8)

    return score

import torch
def get_best(density, pos_embedding, k=5):
# points: [N,3], density: [N]
    # k = 5
    points = torch.stack([p.center for p in pos_embedding])
    # density=density
    N = points.shape[0]

    best_sum = -1
    best_group = None
    best_ndx = None
    scores = {}

    for i in range(N):
        # compute distances from point i
        dists = torch.norm(points - points[i], dim=1)  # [N]
        
        # find indices of the closest k points including self
        _, nn_idx = torch.topk(-dists, k)  # negative because topk returns largest
        
        # sum densities in this group
        group_density = density[nn_idx].sum()
        scores[i] = {}
        scores[i]['group_density'] = group_density
        scores[i]['nn_idx'] = nn_idx
        
        if group_density > best_sum:
            best_sum = group_density
            best_group = nn_idx
            best_ndx = i

    highest_cluster_points = points[best_group]
    highest_cluster_density = density[best_group]

    return scores, points


def embed(xyz2, frame_emb,  step, protein_coords_list,protein_cofactors,global_to_local):
    print("STEP:", step)

    pos_embedding = []
    atom_coords = protein_coords_list
    
    # print(dist)
    min_dist = 2.5
    
    for point in xyz2[0:]:
        center = point
        dist = torch.norm(center - atom_coords[cat[0],:])
        radius = 5.0

        # Compute distances of all atoms to perturbed point
        dists = torch.norm(atom_coords - center, dim=1)

        # Indices of atoms inside sphere
                    # Indices of atoms inside sphere
        inside_indices = protein_cofactors[torch.where(dists <= radius)[0]]

        # Ensure inside_indices is always a 1D tensor
        if torch.is_tensor(inside_indices) and inside_indices.ndim == 0:
            inside_indices = inside_indices.unsqueeze(0)

        if isinstance(inside_indices, (int, np.integer)):
            inside_indices = torch.tensor([inside_indices], device=device)

        if inside_indices.numel() == 0:
            dddd = torch.cdist(center.unsqueeze(0), atom_coords)
            print('too far start or end', dddd.min(), flush=True)
            continue
            
        local_inside_indices = np.array([global_to_local[int(g)] for g in inside_indices])

        atom_matrix=types_array_atom[inside_indices]
        positions=atom_coords[local_inside_indices]
        node_directions = positions - atom_coords[cat[0],:]
        node_distances = torch.norm(node_directions, dim=1)
        local_pos_normalized = (positions - center)/5  # shape [N, 3]
        
        test = extract_point_cloud(atom_matrix, positions, center)
        # test.x=torch.column_stack([test.x, torch.tensor(frame_emb[idx]).repeat(len(test.x))])
        frame_vector = frame_emb[step]          # [16]
        frame_vector = frame_vector.unsqueeze(0)           # [1, 16]
        frame_vector = frame_vector.expand(len(test.x), -1)  # [N, 16]
        test.distance = dist.expand(len(test.x))
        test.x = torch.cat([test.x, frame_vector], dim=1)    # [N, 6 + 16]

        test.edge_index, test.edge_attr = build_edges_and_attrs(inside_indices)
        test.inside_indices = inside_indices
        test.center = center
        test.node_attr = local_pos_normalized
        test.node_directions = node_directions
        test.node_distances = node_distances
        pos_embedding.append(test)
    return pos_embedding

def topk_with_radius(density, pos_embedding, k=5, radius=2.0):

    points = torch.stack([p.center for p in pos_embedding])
    N = points.shape[0]
    best_sum = -1
    best_group = None
    best_ndx = None
    scores = {}

    for i in range(N):
        # distances from point i
        dists = torch.norm(points - points[i], dim=1)

        # initial K nearest indices (including itself)
        _, nn_idx = torch.topk(-dists, k)

        # extract positions of candidate group
        group_pts = points[nn_idx]            # [k,3]

        # compute pairwise distance matrix within group
        pdist = torch.norm(
            group_pts.unsqueeze(1) - group_pts.unsqueeze(0),
            dim=2
        )  # [k, k]
        # check radius requirement: all pairwise distances ≤ radius
        if (pdist <= radius).all():
            group_density = density[nn_idx].sum()

            if group_density > best_sum:
                best_sum = group_density
                best_group = nn_idx
                best_ndx = i
            scores[i] = {}
            scores[i]['group_density'] = group_density
            scores[i]['nn_idx'] = nn_idx

    return scores, points

# Map from global RDKit atom idx → local 0..N-1
import itertools

def build_edges_and_attrs(inside_indices):
    atom_to_local = {int(a): i for i, a in enumerate(inside_indices.tolist())}

        # --- Extract bonded edges from RDKit ---
    bonded_edges = []
    bonded_attrs = []

    for bond in mol.GetBonds():
        a1, a2 = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()

        # Only keep if both atoms are in the subset
        if a1 in atom_to_local and a2 in atom_to_local and a1 not in gas_atoms and a2 not in gas_atoms:
            i1, i2 = atom_to_local[a1], atom_to_local[a2]

            bt = bond.GetBondType()
            if bt == Chem.rdchem.BondType.SINGLE:
                order = 1
            elif bt == Chem.rdchem.BondType.DOUBLE:
                order = 2
            elif bt == Chem.rdchem.BondType.AROMATIC:
                order = 3
            else:
                order = 1

            # undirected edges
            bonded_edges += [[i1, i2], [i2, i1]]
            bonded_attrs += [order, order]

    for (a1 ,a2) in gas_atoms:
        if a1 in atom_to_local and a2 in atom_to_local:
            i1, i2 = atom_to_local[a1], atom_to_local[a2]
            bonded_edges += [[i1, i2], [i2, i1]]
            bonded_attrs += [1, 1]

    # --- Build nonbonded edges among all pairs in subset ---
    N = len(inside_indices)
    all_pairs = list(itertools.combinations(range(N), 2))

    bonded_set = set(tuple(sorted(e)) for e in [(a, b) for a, b in bonded_edges if a < b])

    nonbonded_edges = []
    nonbonded_attrs = []

    for i, j in all_pairs:
        if (i, j) not in bonded_set:
            nonbonded_edges += [[i, j], [j, i]]
            nonbonded_attrs += [0, 0]  # edge_attr = 0 for nonbonded

    # --- Combine everything ---
    # edge_index = torch.tensor(bonded_edges + nonbonded_edges, dtype=torch.long).T
    edges = bonded_edges + nonbonded_edges  # list of [i, j] pairs
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(bonded_attrs + nonbonded_attrs, dtype=torch.long)

    return edge_index, edge_attr.unsqueeze(1)

import torch
import math

def predict(pos_embedding, model):
    # model = torch.load('../Sim1/model.pt') # 
    # model=torch.load("../Sim5/model%d_FE2dis_inr6.pt"%24)
    model=torch.load(model)
    device='cuda'
    model = model.to(device)


    test_loader = DataLoader(pos_embedding, batch_size=3, shuffle=False) 


    values = []
    with torch.no_grad():
        for batch_idx, (data_list) in enumerate(test_loader):
            if (batch_idx + 1) % 20 == 0:
                print("Batch",batch_idx+1, flush=True)
            if isinstance(data_list, list):
                from torch_geometric.data import Batch
                batch = Batch.from_data_list(data_list)
            else:
                batch = data_list  # if batch_size=1, it might already be a Data object

            batch = batch.to(device)
            

            
            distance = torch.norm(batch.node_attr[:, :3], dim=1, keepdim=True)  # [N, 1]

            # 2. Convert Cartesian vectors to irreps vector
            x = CartesianTensor("i")
            vector_irrep = x.from_cartesian(batch.node_attr[:, :3])  # [N, 3]

          
            atom_type_onehot = batch.x[:, 0:6]
            frame_emb = batch.x[:, 6:] 
            # node_attr = torch.cat([distance, vector_irrep, frame_emb], dim=1)
            node_attr = torch.cat([
                distance,            # 1 scalar (0e)
                atom_type_onehot,    # 6 scalars (0e)
                frame_emb,           # k scalars (0e)
                vector_irrep,        # 3-vector (1o)
            ], dim=1)
            
            # node_input = torch.ones((batch.num_nodes, 1), device=batch.x.device)
            node_input = torch.cat([
                # batch.distance.unsqueeze(-1), # 1 scalar (0e) distance of center of graph to iron
                batch.node_distances.unsqueeze(-1),    # 1 scalar (0e) distance to iron for each atom
                F.normalize(batch.node_directions, p=2, dim=1),        # 3-vector (1o) direction to iron for each atom
            ], dim=1)

            data = {
                "batch": batch.batch,
                # "x": batch.x[:,0:6], # atom type
                # "frame_emb": frame,
                "x": node_input,
                "node_attr": node_attr, 
                "edge_index": batch.edge_index,
                "edge_attr": batch.edge_attr,
                "pos": batch.pos,  # if needed in preprocess
            }

            outputs = model(data)
            preds = torch.sigmoid(outputs.squeeze(-1))
            values.append(preds)

        results = torch.concat(values)
        best = results.argmax() # 2552
        best_pos = xyz2[best]
        return best, best_pos, results

def sinusoidal_embedding(frame_idx: torch.Tensor, dim: int = 16):
    """
    frame_idx: tensor of shape [N] with normalized frame values in [0,1]
    dim: embedding dimension (should be even)
    Returns: tensor of shape [N, dim]
    """
    device = frame_idx.device
    N = frame_idx.size(0)
    pe = torch.zeros(N, dim, device=device)

    position = frame_idx.unsqueeze(1)  # [N, 1]
    div_term = torch.exp(torch.arange(0, dim, 2, device=device) * -(math.log(10000.0) / dim))  # [dim/2]

    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe  # [N, dim]

    

In [12]:
# import pandas as pd 

# predicted_paths = pd.read_csv("/media/bw973/Seagate Hub1/Oxygenases/PCO_MUTANTS/ARABI/predict.csv")
# df = predicted_paths[(predicted_paths.name=='PCO4_WT') & (predicted_paths.Sim==2) & (predicted_paths.prop=='counts')]
# df.iloc[:,[0,1,2,3,4,5,6,7,8,277]]

# time=0
# for k, (i, row) in enumerate(df.iloc[0:,].iterrows()):
#     gas_resid=row['Gas_Resid']
#     gas_idx = np.where(residue_sel_un==gas_resid)[0][0]
#     start = int(row['range'].split('-')[0])
#     end = int(row['range'].split('-')[1])
#     gas_resid=row['Gas_Resid']
#     dists = d[start:end+1,gas_idx,0]
#     first = torch.where(dists < 6)[0][0]
#     # print(first+1, end-(start+first))
#     time += (end-start+1)

# time

In [13]:
# import pandas as pd 

# predicted_paths = pd.read_csv("/media/bw973/Seagate Hub1/Oxygenases/PCO_MUTANTS/ARABI/predict.csv")
# df = predicted_paths[(predicted_paths.name=='PCO4_WT') & (predicted_paths.Sim==2) & (predicted_paths.prop=='counts')]
# df.iloc[:,[0,1,2,3,4,5,6,7,8,277]]

In [48]:
meet146 = traj.topology.select("resname MET and residue 146 and name SD")
# meet146 = traj.topology.select("element S")
meet146

array([2261])

In [13]:
gas_idx = np.where(residue_sel_un==250000)[0]
gas_idx
protein_cofactors = torch.tensor(np.setdiff1d(np.arange(0,xyz.shape[1]), gas_atoms[gas_idx])).to(device)
global_to_local = {int(g): i for i, g in enumerate(protein_cofactors)}
global_to_local

{0: 0,
 1: 1,
 2: 2,
 3: 3,
 4: 4,
 5: 5,
 6: 6,
 7: 7,
 8: 8,
 9: 9,
 10: 10,
 11: 11,
 12: 12,
 13: 13,
 14: 14,
 15: 15,
 16: 16,
 17: 17,
 18: 18,
 19: 19,
 20: 20,
 21: 21,
 22: 22,
 23: 23,
 24: 24,
 25: 25,
 26: 26,
 27: 27,
 28: 28,
 29: 29,
 30: 30,
 31: 31,
 32: 32,
 33: 33,
 34: 34,
 35: 35,
 36: 36,
 37: 37,
 38: 38,
 39: 39,
 40: 40,
 41: 41,
 42: 42,
 43: 43,
 44: 44,
 45: 45,
 46: 46,
 47: 47,
 48: 48,
 49: 49,
 50: 50,
 51: 51,
 52: 52,
 53: 53,
 54: 54,
 55: 55,
 56: 56,
 57: 57,
 58: 58,
 59: 59,
 60: 60,
 61: 61,
 62: 62,
 63: 63,
 64: 64,
 65: 65,
 66: 66,
 67: 67,
 68: 68,
 69: 69,
 70: 70,
 71: 71,
 72: 72,
 73: 73,
 74: 74,
 75: 75,
 76: 76,
 77: 77,
 78: 78,
 79: 79,
 80: 80,
 81: 81,
 82: 82,
 83: 83,
 84: 84,
 85: 85,
 86: 86,
 87: 87,
 88: 88,
 89: 89,
 90: 90,
 91: 91,
 92: 92,
 93: 93,
 94: 94,
 95: 95,
 96: 96,
 97: 97,
 98: 98,
 99: 99,
 100: 100,
 101: 101,
 102: 102,
 103: 103,
 104: 104,
 105: 105,
 106: 106,
 107: 107,
 108: 108,
 109: 109,
 110: 110,

In [14]:
print(xyz.shape[1]-1)
global_to_local[xyz.shape[1]-1]

3990


3990

In [9]:
step=0
start=0
frame_pos_bests = []

In [12]:
colors = ['orange','yellow','blue','green','black']

In [10]:

colors = ['blue','green','black','orange','yellow']

In [11]:
start=0
end=100
frames = torch.arange(0, end, dtype=torch.float32)  # [0,1,...,N-1]
frame_norm = frames / (end - start)    
frame_emb = sinusoidal_embedding(frame_norm, dim=16)

In [21]:
                                                         

# end=len(traj)
# frames = torch.arange(0, end, dtype=torch.float32)  # [0,1,...,N-1]
# frame_norm = frames / (end - start)    
# frame_emb = sinusoidal_embedding(frame_norm, dim=16) 
s=0.75


frame_pos=torch.tensor([0,0,0])
# start=19900
start=19900
# start=8
step=0
# frame_pos=torch.tensor([4.75, 12.5, 2.0])
for start in range(start+0,start+1):

    print("START:", start)
    protein_coords_traj = xyz[start,protein_cofactors,:]
    protein_coords_list = protein_coords_traj
    translated = protein_coords_list - protein_coords_list.cpu()[cat].numpy() 
    new_traj = md.Trajectory(xyz=(translated/10).numpy(), topology=traj.topology)
    new_traj.save('translated_frame%d_O2IF.pdb'%start)

    x_edges = np.arange(-20, 21, 1)
    y_edges = np.arange(-20, 21, 1)
    z_edges = np.arange(-20, 21, 1)

    X, Y, Z = np.meshgrid(x_edges, y_edges, z_edges, indexing='ij')

    # Stack the arrays to create a 3D array of shape (N, 3)
    points_3d = torch.tensor(np.stack([X, Y, Z], axis=-1).reshape(-1, 3), dtype=torch.float32)

    translated_ang = translated.clone()
    exclude_points = translated_ang


    threshold = 2.5
    threshold2 = 3.5

    # Compute distances from every point in `points_3d` to every `exclude_point`
    distances=torch.cdist(points_3d, exclude_points)

    # Find points in `points_3d` with all distances >= threshold
    mask = torch.all(distances >= threshold, axis=1)
    mask2 = torch.any(distances < threshold2, axis=1)

    # Filter `points_3d` to keep only points outside the threshold
    filtered_points = points_3d[mask & mask2]

    filtered_points2 = filtered_points# + protein_coords_list.cpu()[cat].numpy() 
    filtered_points2.mean(axis=0)
    filtered_points2.shape
    dist2 = torch.norm(filtered_points2-frame_pos, dim=1)
    if start%100 == 0 or start==9999:
        filtered_points3 = filtered_points2[(dist2 <= 22) & (dist2 >= 10)]
    else:
        filtered_points3 = filtered_points2[dist2 <= 6]
    xyz2=filtered_points3
    print(xyz2.shape)
    # embed_predict and get highest next point to be the next frame_pos at 2...end
    pos_embedding = embed(xyz2, frame_emb, step=step, protein_coords_list=translated_ang,protein_cofactors=protein_cofactors,global_to_local=global_to_local)


    index, frame_pos__, results = predict(pos_embedding, model='../Sim5/model25_FE2dis_inrPmP4_far.pt')




START: 19900
torch.Size([4561, 3])
STEP: 0
Batch 20
Batch 40
Batch 60
Batch 80
Batch 100
Batch 120
Batch 140
Batch 160
Batch 180
Batch 200
Batch 220
Batch 240
Batch 260
Batch 280
Batch 300
Batch 320
Batch 340
Batch 360
Batch 380
Batch 400
Batch 420
Batch 440
Batch 460
Batch 480
Batch 500
Batch 520
Batch 540
Batch 560
Batch 580
Batch 600
Batch 620
Batch 640
Batch 660
Batch 680
Batch 700
Batch 720
Batch 740
Batch 760
Batch 780
Batch 800
Batch 820
Batch 840
Batch 860
Batch 880
Batch 900
Batch 920
Batch 940
Batch 960
Batch 980
Batch 1000
Batch 1020
Batch 1040
Batch 1060
Batch 1080
Batch 1100
Batch 1120
Batch 1140
Batch 1160
Batch 1180
Batch 1200
Batch 1220
Batch 1240
Batch 1260
Batch 1280
Batch 1300
Batch 1320
Batch 1340
Batch 1360
Batch 1380
Batch 1400
Batch 1420
Batch 1440
Batch 1460
Batch 1480
Batch 1500
Batch 1520


In [19]:
# index, frame_pos__, results = predict(pos_embedding, model='../Sim5/model25_FE2dis_inrPmP4_far.pt')
new_traj = md.Trajectory(xyz=(translated/10).numpy(), topology=traj.topology)
new_traj.save('translated_frame%d_O2IF.pdb'%start)

In [11]:
results.mean()

tensor(0.7164, device='cuda:0')

In [48]:
index, frame_pos__, results = predict(pos_embedding, model='../Sim5/model27_FE2dis_inrAll_far.pt')

Batch 20
Batch 40
Batch 60
Batch 80
Batch 100
Batch 120
Batch 140
Batch 160
Batch 180
Batch 200
Batch 220
Batch 240
Batch 260
Batch 280
Batch 300
Batch 320
Batch 340
Batch 360
Batch 380
Batch 400
Batch 420
Batch 440


In [18]:
write_pdb2(torch.arange(0,len(filtered_points3)),results,filtered_points3[:,0],filtered_points3[:,1],filtered_points3[:,2], file=str(start)+ '_frame' + str(step) + '_of_' + str((end)) + 'testO2IF.pdb')

9900_frame0_of_100testO2IF.pdb


In [24]:
def topk_with_radius(density, pos_embedding, k=5, radius=2.0):

    points = torch.stack([p.center for p in pos_embedding])
    N = points.shape[0]
    best_sum = -1
    best_group = None
    best_ndx = None
    scores = {}

    for i in range(N):
        # distances from point i
        dists = torch.norm(points - points[i], dim=1)

        # initial K nearest indices (including itself)
        _, nn_idx = torch.topk(-dists, k)

        # extract positions of candidate group
        group_pts = points[nn_idx]            # [k,3]

        # compute pairwise distance matrix within group
        pdist = torch.norm(
            group_pts.unsqueeze(1) - group_pts.unsqueeze(0),
            dim=2
        )  # [k, k]

        # mask diagonal to ignore 0-distance self-pairs
        # pdist = pdist + torch.eye(k, device=pdist.device) * 999

        # check radius requirement: all pairwise distances ≤ radius
        if (pdist <= radius).all():
            group_density = density[nn_idx].sum()

            if group_density > best_sum:
                best_sum = group_density
                best_group = nn_idx
                best_ndx = i
            scores[i] = {}
            scores[i]['group_density'] = group_density
            scores[i]['nn_idx'] = nn_idx

    return scores, points

scores, points = topk_with_radius(results, pos_embedding, k=25, radius=5.0)

In [28]:
top10 = sorted(scores.items(), key=lambda x: x[1]['group_density'], reverse=True)[0:200]

for key, vals in top10:
    frame_pos=points[vals['nn_idx']].mean(axis=0)
    # print(key, vals['group_density'], points[vals['nn_idx']].mean(axis=0))
    print("draw sphere {",frame_pos.numpy()[0],frame_pos.numpy()[1],frame_pos.numpy()[2],"} radius 1")

draw sphere { -16.08 -10.08 5.12 } radius 1
draw sphere { -15.88 -10.4 5.16 } radius 1
draw sphere { -16.72 0.52 11.64 } radius 1
draw sphere { -16.68 0.28 11.4 } radius 1
draw sphere { -16.4 -9.84 5.0 } radius 1
draw sphere { -14.6 3.16 -3.52 } radius 1
draw sphere { -12.88 -2.16 -7.64 } radius 1
draw sphere { -13.0 -2.24 -7.84 } radius 1
draw sphere { -16.2 -10.16 5.08 } radius 1
draw sphere { -12.8 -2.2 -7.16 } radius 1
draw sphere { -12.76 -2.76 -7.64 } radius 1
draw sphere { -15.88 -10.8 5.08 } radius 1
draw sphere { -16.76 0.44 11.6 } radius 1
draw sphere { 6.48 -15.68 -0.16 } radius 1
draw sphere { -15.72 -10.96 5.2 } radius 1
draw sphere { -16.8 0.28 11.48 } radius 1
draw sphere { -13.04 -2.16 -7.16 } radius 1
draw sphere { 6.68 -15.76 0.32 } radius 1
draw sphere { -17.04 1.16 11.4 } radius 1
draw sphere { -16.32 -10.12 4.28 } radius 1
draw sphere { 14.88 4.96 -12.28 } radius 1
draw sphere { -14.52 3.12 -4.12 } radius 1
draw sphere { -13.16 -2.12 -7.76 } radius 1
draw sphere { 